# Set Up

## Mount Google Drive

Ignore if not using Google Collab:

In [1]:
from google.colab import drive

# mount google drive
drive.mount('/content/drive')
%cd /content/drive/My Drive
!git clone https://github.com/FranciscoLozCoding/cooling_with_code.git
%cd cooling_with_code
!git pull

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive
fatal: destination path 'cooling_with_code' already exists and is not an empty directory.
/content/drive/My Drive/cooling_with_code
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 401 bytes | 7.00 KiB/s, done.
From https://github.com/FranciscoLozCoding/cooling_with_code
   e002b41..6135ed9  main       -> origin/main
Updating e002b41..6135ed9
Fast-forward
 tools/build_dataset.py | 1 +
 1 file changed, 1 insertion(+)
fatal: cannot exec '.git/hooks/post-merge': Permission denied


## Import Libraries

Download libraries not in google collab (can be disregarded if not using collab)

In [2]:
%pip install stackstac
%pip install pystac-client
%pip install planetary-computer
%pip install odc-stac
%pip install rioxarray
%pip install geopandas
%pip install geopy
%pip install folium

In [3]:
#data science
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
import pickle

#custom tools for this project
from tools.environment import VALID_SPLIT, RANDOM_STATE
from tools.build_dataset import (
    generate_buffer_dataset,
    generate_median,
    generate_building_gdf,
    generate_traffic,
    generate_weather_data
)

# Generating a 200m Buffer Dataset

This notebook is for generating new datasets using Buffer Zones between 200m and 400m. After doing [04_EDA](04_EDA.ipynb) we found that increasing the buffer zone from 150m to a value between 200m and 400m it might help our models capture the relationship between vegeatation and UHI better. For details on the features and how we generate our datasets see our past notebooks:
- [01_dataset_generation](01_dataset_generation.ipynb)
- [02_more_dataset_generation](02_more_dataset_generation.ipynb)

>NOTE: we will use our custom tools here, for a in-depth explanation of these tools see the notebooks above.

In [4]:
buffer_radius = 100  # radius in meters (diameter will be 200)

## Training Dataset

We will first create the training dataset.

In [5]:
# Generate the satellite image median.
median = generate_median()

# Generate the building geodataframe.
buildings_gdf = generate_building_gdf()

# Generate traffic data for UHI geodataframe.
uhi_gdf = generate_traffic()

# Read values into a series
uhi = uhi_gdf['UHI Index'].values
traffic_volume = uhi_gdf['traffic_volume'].values
latitudes = uhi_gdf['Latitude'].values
longitudes = uhi_gdf['Longitude'].values
datetimes = uhi_gdf['datetime'].values

# Apply Buffer Zone
train_df = generate_buffer_dataset(
    latitudes, longitudes,
    buffer_radius, traffic_volume,
    median, buildings_gdf,
    UHI=uhi, datetimes=datetimes)

# Add the weather data
train_df = generate_weather_data(train_df)

# remove cols we dont need
cols = ['Latitude', 'Longitude', 'datetime']
train_df.drop(cols, axis=1, inplace=True)

/usr/local/lib/python3.11/dist-packages/pystac_client/item_search.py:881: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


This is the number of scenes that touch our region: 7


/usr/local/lib/python3.11/dist-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


geodecoded_file already exists, loading it...


Processing points: 100%|██████████| 11229/11229 [1:06:12<00:00,  2.83it/s]


In [6]:
# show features
train_df.describe()

,UHI,NDVI,NDBI,NDWI,SI,NDMI,NPCRI,Coastal_Aerosol,Building_Count,Total_Building_Area_m2,Building_Density,Building_Height,Building_Construction_Year,Ground_Elevation,Traffic_Volume,Air Temp at Surface [degC],Relative Humidity [percent],Avg Wind Speed [m/s],Wind Direction [degrees],Solar Flux [W/m^2]
count,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000
mean,1.000001,0.227774,0.006602,-0.243635,0.250644,-0.006602,0.077614,1161.390120,31.224864,9757.523688,0.311091,62.923797,1865.891878,54.028561,98.293381,27.058201,46.813050,3.009561,156.239627,442.933921
std,0.016238,0.156575,0.084050,0.130636,0.107305,0.084050,0.022885,296.150999,22.492268,4128.664710,0.131631,44.061097,366.196457,38.243908,117.881537,0.363503,1.840233,0.075730,17.720758,2.309757
min,0.956122,-0.009321,-0.308972,-0.740815,0.063762,-0.225595,-0.077581,356.348585,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.016216,26.753846,44.615385,2.946154,135.076923,441.000000
25%,0.988577,0.127385,-0.021278,-0.291888,0.189149,-0.060426,0.072183,999.337342,14.000000,7346.208556,0.234213,38.587597,1920.606061,25.090909,33.901786,26.753846,44.615385,2.946154,135.076923,441.000000
50%,1.000237,0.182277,0.030276,-0.207425,0.218396,-0.030276,0.081181,1171.033694,28.000000,10303.822675,0.328508,54.222932,1933.600000,45.558140,64.351533,26.753846,48.353846,2.946154,171.076923,441.000000
75%,1.011176,0.283417,0.060426,-0.161119,0.280770,0.021278,0.089593,1347.215840,44.000000,12704.645038,0.405052,76.916461,1949.125000,78.904762,125.536458,27.492308,48.353846,3.100000,171.076923,445.692308
max,1.046036,0.812449,0.225595,-0.009059,0.705706,0.308972,0.134091,2151.502401,129.000000,22068.139186,0.703580,525.000000,2015.750000,246.047619,2700.284404,27.492308,48.353846,3.100000,171.076923,445.692308


In [7]:
# Save to csv file
train_df.to_csv(f"{buffer_radius*2}m_buffer_dataset.csv", index=False)

## Test Dataset



Now, we will create our testing dataset.

In [8]:
#csv path for target variable for testing dataset
test_csv = "data/Testing_data_uhi_index.csv"

# Generate traffic data for UHI geodataframe.
test_uhi_df = generate_traffic(uhi_csv_file="data/Testing_data_uhi_index.csv")

# Read values into a series
traffic_volume = test_uhi_df['traffic_volume'].values
latitudes = test_uhi_df['Latitude'].values
longitudes = test_uhi_df['Longitude'].values

# Apply Buffer Zone
test_df = generate_buffer_dataset(
    latitudes, longitudes,
    buffer_radius, traffic_volume,
    median, buildings_gdf)

# Add the weather data
test_df = generate_weather_data(test_df)

# drop variables we dont need
# NOTE: We need the lat and lon here, since they are required in the final submission
cols = ['datetime', 'UHI']
test_df.drop(cols, axis=1, inplace=True)

geodecoded_file already exists, loading it...


Processing points: 100%|██████████| 1040/1040 [04:45<00:00,  3.64it/s]


In [9]:
# show features
test_df.describe()

,Longitude,Latitude,NDVI,NDBI,NDWI,SI,NDMI,NPCRI,Coastal_Aerosol,Building_Count,...,Building_Density,Building_Height,Building_Construction_Year,Ground_Elevation,Traffic_Volume,Air Temp at Surface [degC],Relative Humidity [percent],Avg Wind Speed [m/s],Wind Direction [degrees],Solar Flux [W/m^2]
count,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,...,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000
mean,-73.934816,40.807991,0.227462,0.006206,-0.243955,0.249544,-0.006206,0.078299,1157.461559,31.124038,...,0.314449,63.278550,1863.815781,53.768607,99.775465,27.054911,46.829704,3.008876,156.400000,442.913018
std,0.028661,0.023200,0.153267,0.082315,0.127382,0.104745,0.082315,0.022708,288.953402,21.964689,...,0.132780,42.035855,373.617174,37.640960,118.999329,0.363059,1.837984,0.075637,17.699106,2.306935
min,-73.993163,40.758877,0.025896,-0.305542,-0.726354,0.091020,-0.205399,-0.066357,358.542943,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.016216,26.753846,44.615385,2.946154,135.076923,441.000000
25%,-73.957030,40.790802,0.128800,-0.020488,-0.288755,0.186428,-0.059515,0.072538,999.758816,14.000000,...,0.236938,39.166687,1921.153226,25.228448,32.387649,26.753846,44.615385,2.946154,135.076923,441.000000
50%,-73.934618,40.809553,0.185771,0.029550,-0.210250,0.218186,-0.029550,0.081095,1158.299951,27.000000,...,0.330183,54.844368,1934.927273,45.870813,61.428571,26.753846,48.353846,2.946154,171.076923,441.000000
75%,-73.910655,40.823054,0.283548,0.059515,-0.161254,0.278414,0.020488,0.089882,1332.699337,44.000000,...,0.408100,77.664293,1949.500000,78.122869,125.540039,27.492308,48.353846,3.100000,171.076923,445.692308
max,-73.879537,40.859243,0.794935,0.205399,-0.034062,0.691949,0.305542,0.127429,2012.205900,118.000000,...,0.650800,334.500000,2015.333333,244.954545,855.022569,27.492308,48.353846,3.100000,171.076923,445.692308


In [10]:
# Save to csv file
test_df.to_csv(f"{buffer_radius*2}m_buffer_test_dataset.csv", index=False)

## Evaluating the 200m Buffer Dataset

Finally, let's evaluate how this new dataset does on a simple RandomForestRegressor.

In [11]:
dataset = train_df.copy()

# Scale data using standardscaler
sc = StandardScaler()
scaled_dataset = sc.fit_transform(dataset)

# Convert back to a DataFrame with original columns and index
dataset = pd.DataFrame(scaled_dataset, columns=dataset.columns, index=dataset.index)

# Split the data into features (X) and target (y), and then into training and validation sets
x = dataset.drop(columns=['UHI']).values
y = dataset['UHI'].values
x_train, x_valid, y_train, y_valid = train_test_split(
    x, y,
    test_size=VALID_SPLIT,
    random_state=RANDOM_STATE)
x_names = list(dataset.drop(columns=['UHI']).columns)

# Train the Random Forest model on the training data
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=RANDOM_STATE,
    criterion="squared_error")
rf_model.fit(x_train, y_train)

# Make predictions on the training data
insample_predictions = rf_model.predict(x_train)

# calculate R-squared score for in-sample predictions
print(f"In-Sample Evaluation:")
insample_r2 = r2_score(y_train, insample_predictions)
print(f"  R-squared: {insample_r2}")

# Make predictions on the validation data
out_of_sample_predictions = rf_model.predict(x_valid)

# calculate R-squared score for out-sample predictions
print(f"Out-Of-Sample Evaluation:")
out_of_sample_r2 = r2_score(y_valid, out_of_sample_predictions)
print(f"  R-squared: {out_of_sample_r2}")

In-Sample Evaluation:
  R-squared: 0.9913477717982113
Out-Of-Sample Evaluation:
  R-squared: 0.9401849875163022


Here, we see that our 200m dataset did better than our 150m dataset. Using both on a RandomForestRegressor resulted in our 200m getting a higher r-square. The 150m RF model got an Out-Sample R-squared of 0.92945. Let's try increasing the buffer again to see if it does better.

# Generating a 300m Buffer Dataset

Now, we will see how a 300m Buffer Dataset will do.

In [12]:
buffer_radius = 150  # radius in meters (diameter will be 300)

## Training Dataset

We will first create the training dataset.

In [13]:
# Generate the satellite image median.
median = generate_median()

# Generate the building geodataframe.
buildings_gdf = generate_building_gdf()

# Generate traffic data for UHI geodataframe.
uhi_gdf = generate_traffic()

# Read values into a series
uhi = uhi_gdf['UHI Index'].values
traffic_volume = uhi_gdf['traffic_volume'].values
latitudes = uhi_gdf['Latitude'].values
longitudes = uhi_gdf['Longitude'].values
datetimes = uhi_gdf['datetime'].values

# Apply Buffer Zone
train_df = generate_buffer_dataset(
    latitudes, longitudes,
    buffer_radius, traffic_volume,
    median, buildings_gdf,
    UHI=uhi, datetimes=datetimes)

# Add the weather data
train_df = generate_weather_data(train_df)

# remove cols we dont need
cols = ['Latitude', 'Longitude', 'datetime']
train_df.drop(cols, axis=1, inplace=True)

This is the number of scenes that touch our region: 7
geodecoded_file already exists, loading it...


Processing points: 100%|██████████| 11229/11229 [1:08:57<00:00,  2.71it/s]


In [14]:
# show features
train_df.describe()

,UHI,NDVI,NDBI,NDWI,SI,NDMI,NPCRI,Coastal_Aerosol,Building_Count,Total_Building_Area_m2,Building_Density,Building_Height,Building_Construction_Year,Ground_Elevation,Traffic_Volume,Air Temp at Surface [degC],Relative Humidity [percent],Avg Wind Speed [m/s],Wind Direction [degrees],Solar Flux [W/m^2]
count,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000,11229.000000
mean,1.000001,0.232469,0.003622,-0.246734,0.253535,-0.003622,0.076877,1155.088675,66.318105,22213.852874,0.314767,60.716967,1885.070659,54.653582,98.293381,27.058201,46.813050,3.009561,156.239627,442.933921
std,0.016238,0.147851,0.078877,0.123206,0.102414,0.078877,0.022784,273.694472,44.385384,8495.083676,0.120374,39.085096,309.761602,36.777865,117.881537,0.363503,1.840233,0.075730,17.720758,2.309757
min,0.956122,-0.016633,-0.306184,-0.717572,0.032418,-0.192701,-0.091767,362.945235,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.016216,26.753846,44.615385,2.946154,135.076923,441.000000
25%,0.988577,0.140898,-0.024036,-0.288883,0.194426,-0.054137,0.072584,1013.956638,31.000000,17687.656779,0.250632,38.430818,1920.500000,26.405405,33.901786,26.753846,44.615385,2.946154,135.076923,441.000000
50%,1.000237,0.190917,0.026224,-0.214271,0.223304,-0.026224,0.081801,1173.780807,59.000000,22930.334472,0.324920,53.656843,1933.809917,47.060870,64.351533,26.753846,48.353846,2.946154,171.076923,441.000000
75%,1.011176,0.281811,0.054137,-0.172109,0.281332,0.024036,0.088897,1324.752409,94.000000,28066.352051,0.397696,74.491858,1946.882353,78.429907,125.536458,27.492308,48.353846,3.100000,171.076923,445.692308
max,1.046036,0.787450,0.192701,0.032471,0.681876,0.306184,0.122749,1967.360257,238.000000,45777.027725,0.648654,396.220747,2015.750000,234.711111,2700.284404,27.492308,48.353846,3.100000,171.076923,445.692308


In [15]:
# Save to csv file
train_df.to_csv(f"{buffer_radius*2}m_buffer_dataset.csv", index=False)

## Test Dataset

Now, we will create our testing dataset.

In [16]:
#csv path for target variable for testing dataset
test_csv = "data/Testing_data_uhi_index.csv"

# Generate traffic data for UHI geodataframe.
test_uhi_df = generate_traffic(uhi_csv_file="data/Testing_data_uhi_index.csv")

# Read values into a series
traffic_volume = test_uhi_df['traffic_volume'].values
latitudes = test_uhi_df['Latitude'].values
longitudes = test_uhi_df['Longitude'].values

# Apply Buffer Zone
test_df = generate_buffer_dataset(
    latitudes, longitudes,
    buffer_radius, traffic_volume,
    median, buildings_gdf)

# Add the weather data
test_df = generate_weather_data(test_df)

# drop variables we dont need
# NOTE: We need the lat and lon here, since they are required in the final submission
cols = ['datetime', 'UHI']
test_df.drop(cols, axis=1, inplace=True)

geodecoded_file already exists, loading it...


Processing points: 100%|██████████| 1040/1040 [04:50<00:00,  3.58it/s]


In [17]:
# show features
test_df.describe()

,Longitude,Latitude,NDVI,NDBI,NDWI,SI,NDMI,NPCRI,Coastal_Aerosol,Building_Count,...,Building_Density,Building_Height,Building_Construction_Year,Ground_Elevation,Traffic_Volume,Air Temp at Surface [degC],Relative Humidity [percent],Avg Wind Speed [m/s],Wind Direction [degrees],Solar Flux [W/m^2]
count,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,...,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000,1040.000000
mean,-73.934816,40.807991,0.232582,0.003460,-0.247338,0.253100,-0.003460,0.077530,1152.900844,66.208654,...,0.319175,61.248658,1884.015589,54.329376,99.775465,27.054911,46.829704,3.008876,156.400000,442.913018
std,0.028661,0.023200,0.144867,0.077908,0.120254,0.099427,0.077908,0.022483,267.962783,43.741318,...,0.123230,39.106558,314.237278,36.218207,118.999329,0.363059,1.837984,0.075637,17.699106,2.306935
min,-73.993163,40.758877,0.013546,-0.304482,-0.715038,0.060900,-0.172080,-0.078905,376.290773,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.016216,26.753846,44.615385,2.946154,135.076923,441.000000
25%,-73.957030,40.790802,0.141392,-0.023876,-0.287524,0.196312,-0.052577,0.072941,1017.605076,32.000000,...,0.252918,39.083797,1919.632445,26.217391,32.387649,26.753846,44.615385,2.946154,135.076923,441.000000
50%,-73.934618,40.809553,0.191252,0.027066,-0.214880,0.223840,-0.027066,0.081971,1168.457384,62.000000,...,0.330735,53.823053,1934.256410,47.855682,61.428571,26.753846,48.353846,2.946154,171.076923,441.000000
75%,-73.910655,40.823054,0.280489,0.052577,-0.172633,0.280822,0.023876,0.089412,1315.734731,94.000000,...,0.402295,75.761725,1948.122441,77.900694,125.540039,27.492308,48.353846,3.100000,171.076923,445.692308
max,-73.879537,40.859243,0.783853,0.172080,-0.011880,0.679166,0.304482,0.122888,1931.456592,226.000000,...,0.637365,396.220747,2009.250000,232.387755,855.022569,27.492308,48.353846,3.100000,171.076923,445.692308


In [18]:
# Save to csv file
test_df.to_csv(f"{buffer_radius*2}m_buffer_test_dataset.csv", index=False)

## Evaluating the 300m Buffer Dataset

In [19]:
dataset = train_df.copy()

# Scale data using standardscaler
sc = StandardScaler()
scaled_dataset = sc.fit_transform(dataset)

# Convert back to a DataFrame with original columns and index
dataset = pd.DataFrame(scaled_dataset, columns=dataset.columns, index=dataset.index)

# Split the data into features (X) and target (y), and then into training and validation sets
x = dataset.drop(columns=['UHI']).values
y = dataset['UHI'].values
x_train, x_valid, y_train, y_valid = train_test_split(
    x, y,
    test_size=VALID_SPLIT,
    random_state=RANDOM_STATE)
x_names = list(dataset.drop(columns=['UHI']).columns)

# Train the Random Forest model on the training data
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=RANDOM_STATE,
    criterion="squared_error")
rf_model.fit(x_train, y_train)

# Make predictions on the training data
insample_predictions = rf_model.predict(x_train)

# calculate R-squared score for in-sample predictions
print(f"In-Sample Evaluation:")
insample_r2 = r2_score(y_train, insample_predictions)
print(f"  R-squared: {insample_r2}")

# Make predictions on the validation data
out_of_sample_predictions = rf_model.predict(x_valid)

# calculate R-squared score for out-sample predictions
print(f"Out-Of-Sample Evaluation:")
out_of_sample_r2 = r2_score(y_valid, out_of_sample_predictions)
print(f"  R-squared: {out_of_sample_r2}")

In-Sample Evaluation:
  R-squared: 0.992868531036737
Out-Of-Sample Evaluation:
  R-squared: 0.9477045848852123


Increasing the buffer zone to **300m** improved performance compared to **200m**, but the difference was minimal. The **200m Random Forest model** achieved an **Out-of-Sample R-squared of 0.9401**, indicating strong predictive performance. Given this trend, the performance gains from increasing the buffer zone appear to be leveling off, suggesting that expanding it further may not yield significant improvements.

# Conclusion

Increasing the buffer zone between 200m and 400m did help in improving model accuracy. We first started with 200m and it resulted in a significant increase of R-squared. Then we did 300m and the accuracy did increase but very minimal suggesting the performance gains is leveling off. Because of this we settled with 300m. Any future notebooks will now be using 300m Buffer Zone Dataset.